# 🔮 DỰ ĐOÁN NGUY CƠ RỚT MÔN – DSS AHP + Decision Tree

Nhập `Test_Score`, `Attendance (%)`, `Study_Hours` → Hệ thống trả về:
- **p_fail**: Xác suất rớt từ Decision Tree
- **ahp_score**: Điểm rủi ro có trọng số AHP
- **final_score**: Điểm kết hợp = α × p_fail + (1-α) × ahp_score (mặc định α=0.0 → pure AHP)
- **Mức rủi ro**: Cao (≥0.61) / Trung bình (≥0.31) / Thấp (<0.31)
- **Gợi ý can thiệp**

**Yêu cầu:** Phải chạy `train_model.ipynb` trước để tạo file mô hình `decision_tree_model.pkl`

## 1. Tải mô hình

In [1]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import joblib
import pandas as pd
import numpy as np

from modules.ahp import calculate_weights
from modules.risk_engine import (normalize_to_risk_scalar, calculate_final_score,
                                 classify_risk_level, generate_warning_reason,
                                 suggest_intervention)
from modules.ml_model import get_decision_path_for_student

MODEL_PATH = os.path.join(os.getcwd(), '..', 'models', 'decision_tree_model.pkl')
model = joblib.load(MODEL_PATH)
features = ["Test_Score", "Attendance (%)", "Study_Hours"]

# AHP weights (giống file train)
ahp_matrix = np.array([[1, 3, 5], [1/3, 1, 3], [1/5, 1/3, 1]])
ahp_weights, _, _, cr = calculate_weights(ahp_matrix)

print(f"✅ Đã tải mô hình Decision Tree (nhị phân)")
print(f"   Độ sâu: {model.get_depth()}, Số lá: {model.get_n_leaves()}")
print(f"   AHP weights: {dict(zip(features, np.round(ahp_weights, 3)))}")
print(f"   CR = {cr:.4f}")

✅ Đã tải mô hình Decision Tree (nhị phân)
   Độ sâu: 2, Số lá: 3
   AHP weights: {'Test_Score': np.float64(0.637), 'Attendance (%)': np.float64(0.258), 'Study_Hours': np.float64(0.105)}
   CR = 0.0332


## 2. Hàm dự đoán DSS (AHP + Decision Tree)

In [2]:
def predict_student_dss(test_score, attendance, study_hours, alpha=0.0):
    """
    Dự đoán nguy cơ rớt môn kết hợp AHP + Decision Tree.
    Trả về: p_fail, ahp_score, final_score, mức rủi ro, rule path, gợi ý can thiệp.
    Mặc định alpha=0.0 (pure AHP) – giống hệ thống production.
    """
    X = pd.DataFrame([[test_score, attendance, study_hours]], columns=features)

    # 1. Decision Tree → p_fail
    p_fail = model.predict_proba(X)[0][1]

    # 2. AHP → ahp_score
    risk_values = normalize_to_risk_scalar(test_score, attendance, study_hours)
    ahp_score = sum(ahp_weights[i] * risk_values[features[i]] for i in range(len(features)))
    ahp_score = float(np.clip(ahp_score, 0, 1))

    # 3. Final score = alpha * p_fail + (1-alpha) * ahp_score
    final_score = float(np.clip(alpha * p_fail + (1 - alpha) * ahp_score, 0, 1))

    # 4. Mức rủi ro (ngưỡng: >=0.61 Cao, >=0.31 TB, <0.31 Thấp)
    risk_level = classify_risk_level(final_score)

    # 5. Rule path
    rules = get_decision_path_for_student(model, X, features)

    # 6. Top factors
    top_factors = sorted(risk_values.items(), key=lambda x: ahp_weights[features.index(x[0])] * x[1], reverse=True)
    top_factors_weighted = [(k, ahp_weights[features.index(k)] * v) for k, v in top_factors]

    # 7. Gợi ý can thiệp
    reason = generate_warning_reason(top_factors_weighted[:3])
    suggestion = suggest_intervention(risk_level, top_factors_weighted[:3])

    # In kết quả
    icon = {"Cao": "🔴", "Trung bình": "🟡", "Thấp": "🟢"}.get(risk_level, "⚪")

    print("=" * 60)
    print("  KẾT QUẢ DỰ ĐOÁN – DSS AHP + DECISION TREE")
    print("=" * 60)
    print(f"  Điểm kiểm tra (Test_Score):    {test_score}")
    print(f"  Tỷ lệ đi học (Attendance):     {attendance}%")
    print(f"  Giờ học/ngày (Study_Hours):     {study_hours} giờ")
    print("-" * 60)
    print(f"  📊 p_fail (DT):       {p_fail:.4f}")
    print(f"  ⚖️ ahp_score:         {ahp_score:.4f}")
    print(f"  📐 final_score:       {final_score:.4f}")
    print(f"  {icon} Mức rủi ro:        {risk_level}")
    print("-" * 60)
    print(f"  🌲 Rule Path:")
    for i, r in enumerate(rules, 1):
        print(f"     Bước {i}: {r}")
    print("-" * 60)
    if risk_level in ("Cao", "Trung bình"):
        print(f"  ❗ Lý do cảnh báo:")
        print(f"  {reason}")
        print(f"\n  💡 Gợi ý can thiệp:")
        print(f"  {suggestion}")
    else:
        print(f"  ✅ Sinh viên không có nguy cơ rớt môn.")
    print("=" * 60)

    return {
        "p_fail": p_fail, "ahp_score": ahp_score,
        "final_score": final_score, "risk_level": risk_level,
        "rule_path": rules, "warning_reason": reason,
        "intervention": suggestion,
    }

## 3. Test nhiều trường hợp

In [3]:
test_cases = [
    ("SV giỏi", 8.5, 95, 6),
    ("SV trung bình", 5.0, 70, 3),
    ("SV yếu điểm", 3.0, 80, 4),
    ("SV vắng nhiều", 6.0, 40, 5),
    ("SV lười học", 4.5, 60, 1),
    ("SV nguy hiểm", 2.5, 35, 1.5),
]

results = []
for name, test, attend, hours in test_cases:
    X = pd.DataFrame([[test, attend, hours]], columns=features)
    p_fail = model.predict_proba(X)[0][1]
    risk_vals = normalize_to_risk_scalar(test, attend, hours)
    ahp_score = sum(ahp_weights[i] * risk_vals[features[i]] for i in range(len(features)))
    ahp_score = float(np.clip(ahp_score, 0, 1))
    final = float(np.clip(0.0 * p_fail + 1.0 * ahp_score, 0, 1))
    level = classify_risk_level(final)
    icon = {"Cao": "🔴", "Trung bình": "🟡", "Thấp": "🟢"}.get(level, "⚪")

    results.append({
        "Trường hợp": name,
        "Điểm": test, "Đi học (%)": attend, "Giờ học": hours,
        "p_fail": round(p_fail, 3),
        "ahp_score": round(ahp_score, 3),
        "final_score": round(final, 3),
        "Mức rủi ro": f"{icon} {level}",
    })

pd.DataFrame(results)

,Trường hợp,Điểm,Đi học (%),Giờ học,p_fail,ahp_score,final_score,Mức rủi ro
0,SV giỏi,8.5,95,6.0,0.0,0.161,0.161,🟢 Thấp
1,SV trung bình,5.0,70,3.0,0.0,0.475,0.475,🟡 Trung bình
2,SV yếu điểm,3.0,80,4.0,1.0,0.567,0.567,🟡 Trung bình
3,SV vắng nhiều,6.0,40,5.0,0.0,0.471,0.471,🟡 Trung bình
4,SV lười học,4.5,60,1.0,0.0,0.550,0.550,🟡 Trung bình
5,SV nguy hiểm,2.5,35,1.5,1.0,0.737,0.737,🔴 Cao


## 4. Nhập điểm để dự đoán (DSS AHP + Decision Tree)
Thay đổi giá trị `test_score`, `attendance`, `study_hours` bên dưới rồi chạy cell

In [4]:
# ===== THAY ĐỔI GIÁ TRỊ Ở ĐÂY =====
test_score  = 5.0    # Điểm kiểm tra giữa kỳ (0-10)
attendance  = 80.0   # Tỷ lệ đi học (%) (0-100)
study_hours = 3.0    # Số giờ học trung bình/ngày
# ======================================

result = predict_student_dss(test_score, attendance, study_hours)

  KẾT QUẢ DỰ ĐOÁN – DSS AHP + DECISION TREE
  Điểm kiểm tra (Test_Score):    5.0
  Tỷ lệ đi học (Attendance):     80.0%
  Giờ học/ngày (Study_Hours):     3.0 giờ
------------------------------------------------------------
  📊 p_fail (DT):       0.0000
  ⚖️ ahp_score:         0.4487
  📐 final_score:       0.4487
  🟡 Mức rủi ro:        Trung bình
------------------------------------------------------------
  🌲 Rule Path:
     Bước 1: Test_Score = 5.00 > 3.75
     Bước 2: Study_Hours = 3.00 > 0.75
------------------------------------------------------------
  ❗ Lý do cảnh báo:
  • Điểm kiểm tra thấp (đóng góp: 0.32)
• Số giờ tự học ít (đóng góp: 0.08)
• Tỷ lệ chuyên cần thấp (đóng góp: 0.05)

  💡 Gợi ý can thiệp:
  🟡 CẦN THEO DÕI:
  - Nhắc nhở sinh viên
  - Tư vấn học tập
  - Hỗ trợ ôn tập, phụ đạo kiến thức
  - Hướng dẫn phương pháp tự học
  - Nhắc chuyên cần, kiểm tra lý do vắng
